# FL Aggregator — FedAvg Central Server
**FL-Pneumonia-Detection | Distributed Simulation**

Run this **after all 3 hospital notebooks** have completed for a given round.

### What it does
1. Downloads the 3 hospital weight artifacts from WandB
2. Runs weighted FedAvg aggregation
3. Evaluates the global model on the held-out test set
4. Uploads the new global model as `global-model:round-{ROUND_NUM}`
5. Logs all metrics to WandB

### Before running
- Set `ROUND_NUM` to match what the hospital notebooks just ran
- Verify all 3 hospitals have uploaded their artifacts for this round

In [ ]:
!pip install opacus==1.4.0 wandb --quiet
print('Packages ready')

In [ ]:
import os, json
from collections import OrderedDict
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import wandb
from opacus.validators import ModuleValidator
from kaggle_secrets import UserSecretsClient

# ── UPDATE THIS BEFORE EACH ROUND ──
ROUND_NUM   = 1
# ───────────────────────────────────

NUM_CLIENTS    = 3
DATA_ROOT      = '/kaggle/input/chest-xray-pneumonia/chest_xray/chest_xray'
WORK_DIR       = '/kaggle/working'
WANDB_PROJECT  = 'fl-pneumonia-detection'
GLOBAL_ART     = 'global-model'
HOSPITAL_ART   = 'hospital-weights'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Aggregator | Round {ROUND_NUM} | Device: {device}')

wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
wandb.login(key=wandb_key)
print('WandB logged in')

In [ ]:
# ── Model definition (must match hospital notebooks exactly) ──
def fix_for_opacus(module):
    """Replace BatchNorm2d with GroupNorm recursively (Opacus compatibility).
    Avoids ModuleValidator.fix() which internally uses torch.load and breaks
    with PyTorch 2.6+ (weights_only=True default change)."""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            num_channels = child.num_features
            num_groups = min(32, num_channels)
            while num_channels % num_groups != 0:
                num_groups -= 1
            gn = nn.GroupNorm(num_groups, num_channels,
                              eps=child.eps, affine=child.affine)
            if child.affine:
                gn.weight.data.copy_(child.weight.data)
                gn.bias.data.copy_(child.bias.data)
            setattr(module, name, gn)
        else:
            fix_for_opacus(child)
    return module

def create_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, 2),
    )
    return fix_for_opacus(model)
def federated_avg(state_dicts, n_samples_list):
    """
    Weighted average of state_dicts.
    Each hospital's contribution is proportional to its dataset size.
    """
    total = sum(n_samples_list)
    avg_state = OrderedDict()
    for key in state_dicts[0].keys():
        avg_state[key] = sum(
            sd[key].float() * (n / total)
            for sd, n in zip(state_dicts, n_samples_list)
        )
    return avg_state

print('FedAvg and model helpers defined')

In [ ]:
# ── Download hospital weights from WandB ──────────────────────
api = wandb.Api()

hospital_state_dicts = []
hospital_n_samples   = []
hospital_metadata    = []

for hid in range(NUM_CLIENTS):
    alias = f'h{hid}-round-{ROUND_NUM}'
    print(f'Downloading {HOSPITAL_ART}:{alias}...')

    artifact = api.artifact(f'{WANDB_PROJECT}/{HOSPITAL_ART}:{alias}', type='model')
    art_dir  = artifact.download(root=f'{WORK_DIR}/hospital_{hid}')
    ckpt     = torch.load(
        os.path.join(art_dir, f'hospital_{hid}.pth'), map_location='cpu'
    )

    hospital_state_dicts.append(ckpt['model_state_dict'])
    hospital_n_samples.append(ckpt['n_samples'])
    hospital_metadata.append({
        'hospital_id':    ckpt['hospital_id'],
        'hospital_name':  ckpt['hospital_name'],
        'n_samples':      ckpt['n_samples'],
        'train_loss':     ckpt['train_loss'],
        'train_accuracy': ckpt['train_accuracy'],
        'epsilon':        ckpt['epsilon'],
    })

    print(f'  H{hid} ({ckpt["hospital_name"]}): '
          f'n={ckpt["n_samples"]}  '
          f'acc={ckpt["train_accuracy"]:.4f}  '
          f'eps={ckpt["epsilon"]}')

print(f'\nAll {NUM_CLIENTS} hospital weights downloaded')

In [ ]:
# ── FedAvg aggregation ────────────────────────────────────────
print('Running FedAvg aggregation...')
global_state_dict = federated_avg(hospital_state_dicts, hospital_n_samples)

global_model = create_model()
global_model.load_state_dict(global_state_dict)
global_model = global_model.to(device)
print('FedAvg complete')

In [ ]:
# ── Evaluate on test set ──────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

class ChestXrayDataset(Dataset):
    CLASSES = ['NORMAL', 'PNEUMONIA']
    def __init__(self, data_dir, transform=None):
        self.transform = transform
        self.images, self.labels = [], []
        for label, cls in enumerate(self.CLASSES):
            cls_dir = os.path.join(data_dir, cls)
            if not os.path.isdir(cls_dir): continue
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(('.jpg','.jpeg','.png')):
                    self.images.append(os.path.join(cls_dir, fname))
                    self.labels.append(label)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        return (self.transform(img) if self.transform else img), self.labels[idx]

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

test_ds     = ChestXrayDataset(os.path.join(DATA_ROOT, 'test'), test_transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)
print(f'Test set: {len(test_ds)} images  ({len(test_loader)} batches)')
print('Note: val split = 16 images only — we use test set for all evaluation.')

global_model.eval()
criterion = nn.CrossEntropyLoss()
total_loss, correct, total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs     = global_model(images)
        total_loss += criterion(outputs, labels).item() * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)

test_loss = total_loss / total
test_acc  = correct    / total
print(f'\nGlobal model after round {ROUND_NUM}:')
print(f'  Test accuracy: {test_acc:.4f}')
print(f'  Test loss:     {test_loss:.4f}')

In [ ]:
# ── Log to WandB + upload global model ───────────────────────
avg_train_acc  = np.mean([m['train_accuracy'] for m in hospital_metadata])
avg_train_loss = np.mean([m['train_loss']     for m in hospital_metadata])
eps_vals       = [m['epsilon'] for m in hospital_metadata if m['epsilon']]
max_eps        = max(eps_vals) if eps_vals else None

run = wandb.init(
    project=WANDB_PROJECT,
    name=f'aggregator-round-{ROUND_NUM}',
    job_type='aggregation',
    config={'round': ROUND_NUM, 'num_clients': NUM_CLIENTS},
)

log = {
    'round':               ROUND_NUM,
    'train/avg_loss':      float(avg_train_loss),
    'train/avg_accuracy':  float(avg_train_acc),
    'test/accuracy':       float(test_acc),
    'test/loss':           float(test_loss),
}
if max_eps is not None:
    log['privacy/epsilon_max']        = float(max_eps)
    log['privacy/epsilon_cumulative'] = float(max_eps * ROUND_NUM)

# Per-hospital breakdown
for m in hospital_metadata:
    hid = m['hospital_id']
    log[f'hospital_{hid}/train_accuracy'] = m['train_accuracy']
    log[f'hospital_{hid}/train_loss']     = m['train_loss']
    log[f'hospital_{hid}/n_samples']      = m['n_samples']
    if m['epsilon']:
        log[f'hospital_{hid}/epsilon'] = m['epsilon']

wandb.log(log, step=ROUND_NUM)

# Save and upload global model
global_ckpt_path = f'{WORK_DIR}/global_model_round_{ROUND_NUM}.pth'
torch.save({
    'model_state_dict': global_model.state_dict(),
    'round':            ROUND_NUM,
    'test_accuracy':    test_acc,
    'test_loss':        test_loss,
    'hospital_metadata': hospital_metadata,
    'classes':          ['NORMAL', 'PNEUMONIA'],
}, global_ckpt_path)

artifact = wandb.Artifact(
    name=GLOBAL_ART, type='model',
    metadata={'round': ROUND_NUM, 'test_accuracy': test_acc, 'max_epsilon': max_eps},
)
artifact.add_file(global_ckpt_path, name='global_model.pth')
run.log_artifact(artifact, aliases=[f'round-{ROUND_NUM}', 'latest'])
wandb.finish()

print(f'Global model uploaded: "{GLOBAL_ART}:round-{ROUND_NUM}"')
print(f'\n=== Round {ROUND_NUM} Summary ===')
print(f'Avg train accuracy: {avg_train_acc:.4f}')
print(f'Global test accuracy: {test_acc:.4f}')
if max_eps:
    print(f'Max epsilon this round: {max_eps:.4f}  (cumulative: {max_eps*ROUND_NUM:.4f})')
print(f'\nNext: increment ROUND_NUM to {ROUND_NUM+1} in all notebooks and repeat.')